# Merge the INQUIRE adapter and convert to GGUF (Component 3 viva demo)

A **build step only**: it turns the LoRA adapter from `train_inquire_lora_colab.ipynb` into a file LM Studio can load.
The demo itself runs on the laptop; Colab is not part of it.

```
adapter (Drive) + google/gemma-3-1b-it --merge--> merged HF model --convert--> adapted-f16.gguf
google/gemma-3-1b-it ------------------------------convert--------------------> base-f16.gguf
```

Both GGUFs use the **same base revision, converter and precision (f16)**, so the only difference between the two
arms is the adapter. (The `gemma-3-1B-it-QAT-Q4_0.gguf` that LM Studio serves for INTERCEPT is a different,
quantised file and is not the comparison base.)

**Before you start**
- Runtime: a CPU runtime is enough; a T4 is fine too. The merge runs on CPU in fp32 (about 4 GB RAM).
- Colab secret `HF_TOKEN` for a Hugging Face account that has accepted the Gemma licence.
- The adapter is at `MyDrive/c3_inquire_training/2026-10-10_1249/adapter` (set in the next cell).

**Outputs** go to Drive (`MyDrive/c3_inquire_gguf/<run id>/`): `adapted-f16.gguf`, `base-f16.gguf`, `conversion_record.json`.
Only `conversion_record.json` comes back to the repo (as evidence). The GGUFs never do (`*.gguf` is gitignored).

In [ ]:
!pip -q install "transformers==4.56.2" "peft==0.17.1" "accelerate==1.10.1" sentencepiece protobuf

In [ ]:
import json, time
from google.colab import drive
drive.mount("/content/drive")

TRAIN_RUN_ID = "2026-10-10_1249"
ADAPTER_DIR = f"/content/drive/MyDrive/c3_inquire_training/{TRAIN_RUN_ID}/adapter"
BASE_ID = "google/gemma-3-1b-it"
BASE_REVISION = "dcc83ea841ab6100d6b47a070329e1ba4cf78752"   # from the training run_record.json

RUN_ID = time.strftime("%Y-%m-%d_%H%M", time.gmtime())
OUT_DIR = f"/content/drive/MyDrive/c3_inquire_gguf/{RUN_ID}"
WORK = "/content/work"
!mkdir -p "{OUT_DIR}" "{WORK}"

cfg = json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))
print("adapter base:", cfg["base_model_name_or_path"], "| r =", cfg["r"], "| alpha =", cfg["lora_alpha"])
assert cfg["base_model_name_or_path"] == BASE_ID, "adapter was trained on a different base"
print("output directory:", OUT_DIR)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 1. Merge (fp32, CPU)
fp32 matches how the adapter was trained. The merge adds the LoRA deltas into the base weights.

In [ ]:
import torch, transformers, peft
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_ID, revision=BASE_REVISION)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, revision=BASE_REVISION, torch_dtype=torch.float32)
base.save_pretrained(f"{WORK}/base_hf"); tok.save_pretrained(f"{WORK}/base_hf")   # for the control GGUF

merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(f"{WORK}/merged_hf", safe_serialization=True); tok.save_pretrained(f"{WORK}/merged_hf")
print("merged and saved")

## 2. Convert both to GGUF (f16) with llama.cpp
The llama.cpp commit is recorded, so the conversion can be repeated.

In [ ]:
!git clone -q https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp
!pip -q install -r {WORK}/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
LLAMA_CPP_COMMIT = !git -C {WORK}/llama.cpp rev-parse HEAD
LLAMA_CPP_COMMIT = LLAMA_CPP_COMMIT[0]
print("llama.cpp commit:", LLAMA_CPP_COMMIT)

!python {WORK}/llama.cpp/convert_hf_to_gguf.py {WORK}/merged_hf --outtype f16 --outfile {OUT_DIR}/adapted-f16.gguf
!python {WORK}/llama.cpp/convert_hf_to_gguf.py {WORK}/base_hf   --outtype f16 --outfile {OUT_DIR}/base-f16.gguf

## 3. Check the merge before trusting it
Greedy generation from the merged HF model on the same prompt as the base. The adapted reply should differ from
the base reply. Compare these two outputs with what LM Studio gives from `adapted-f16.gguf` on the laptop
(same prompt, temperature 0): they should be near-identical, not necessarily byte-identical (f16 rounding).

In [ ]:
PROMPT = "I was just watching videos"   # replace with a held-out case from the test split to match the laptop check
def reply(model):
    msgs = [{"role": "user", "content": PROMPT}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=60, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

base_ref = AutoModelForCausalLM.from_pretrained(f"{WORK}/base_hf", torch_dtype=torch.float32)
print("BASE   :", reply(base_ref))
print("ADAPTED:", reply(merged))

## 4. Record it

In [ ]:
import hashlib, os
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

record = {
    "run_id": RUN_ID,
    "training_run_id": TRAIN_RUN_ID,
    "adapter_location": ADAPTER_DIR,
    "adapter_config": {k: cfg[k] for k in ("r", "lora_alpha", "target_modules")},
    "base_model": BASE_ID, "base_revision": BASE_REVISION,
    "merge_precision": "fp32", "gguf_precision": "f16",
    "llama_cpp_commit": LLAMA_CPP_COMMIT,
    "library_versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
    "files": {n: {"bytes": os.path.getsize(f"{OUT_DIR}/{n}"), "sha256": sha256(f"{OUT_DIR}/{n}")}
              for n in ("adapted-f16.gguf", "base-f16.gguf")},
    "check_prompt": PROMPT, "check_greedy_base": reply(base_ref), "check_greedy_adapted": reply(merged),
}
json.dump(record, open(f"{OUT_DIR}/conversion_record.json", "w"), indent=2)
print(json.dumps(record, indent=2))

## 5. Download
Download both GGUFs from Drive to the laptop (LM Studio's models folder, e.g. `models/c3/adapted-f16.gguf`) and save
`conversion_record.json` into `evidence/gguf_conversion_<date>/` in the repo. Do **not** add the GGUFs to git.